# CSE 151B — LoRA training v8

Pipeline:
1. Split `public.jsonl` → **926 train** / **200 val** (stratified MCQ + free-response)
2. Build SFT targets as gold `\boxed{...}` only
3. Train LoRA with eval + early stopping
4. Save adapter to Drive

**Runtime:** Restart runtime after the install cell, then run all cells below.

In [ ]:
# Cell 1 — Install dependencies (run once, then restart runtime)
!pip uninstall -y torchao
!pip install -U transformers accelerate peft trl datasets bitsandbytes tqdm

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Imports, paths, seed
import os
import re
import json
import random
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

SEED = 42
VAL_SIZE = 200
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
PROJECT_ROOT = "/content/drive/MyDrive/151B_SP26_Competition"
PUBLIC_PATH = f"{PROJECT_ROOT}/data/public.jsonl"

SPLIT_DIR = Path(f"{PROJECT_ROOT}/data/splits_v8")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_RAW_PATH = SPLIT_DIR / "train_public_926.jsonl"
VAL_RAW_PATH = SPLIT_DIR / "val_public_200.jsonl"
TRAIN_SFT_PATH = SPLIT_DIR / "train_sft.jsonl"
VAL_SFT_PATH = SPLIT_DIR / "val_sft.jsonl"
SPLIT_META_PATH = SPLIT_DIR / "split_ids.json"

OUT_DIR = f"{PROJECT_ROOT}/lora_qwen3_reasoning_v8"
os.makedirs(OUT_DIR, exist_ok=True)

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("PUBLIC_PATH:", PUBLIC_PATH)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# Cell 4 — System prompts (aligned with run_inference.py in repo)
SYSTEM_PROMPT_MATH = """You are a confident expert mathematician. You trust your computations and commit to your answers.

Solve the problem step-by-step, then put your final answer in \\boxed{}. For multi-part problems, put all sub-answers in one box separated by commas: \\boxed{3, 7}.

Your answer must match the grader's expected format exactly. The grader does literal comparison — `7.797` will NOT match `7.79743547584717`, and `1.363` will NOT match `atan(4.76)`. Choose the form carefully:

**Prefer exact/symbolic forms. Do NOT evaluate to a decimal unless the problem explicitly asks for one.**
- Inverse trig: write `atan(4.76)`, not `1.363`
- Constants: write `pi`, not `3.142`; write `sqrt(2)/2`, not `0.707`
- Logarithms: write `ln(11/8)` or `[ln(0.5)]/[ln(0.96584)]`, not their decimals
- Fractional powers: write `(1/2)^(36/31)`, not `0.447`
- Unevaluated products when the problem is about deriving a formula: `325*(1+325)` rather than `105950`
- Fractions: write `5/8`, not `0.625`

**When a decimal IS required, give the FULL unrounded value (12+ significant figures).**
Round only when the problem explicitly says to. Otherwise:
- Write `7.79743547584717`, not `7.797`
- Write `442.857142857143`, not `442.86`
If you compute 565/9, do not stop at "≈ 62.78" — write out 62.7777777777778.

**Match the rounding rule the problem states.**
- "Round to integer" with x = 14.4375 → 14
- "After how many full years until X happens" — if the event happens during year 14 (between t=14 and t=15), the answer is often 13 (full years completed before it happens), not 14. Read carefully.
- "How many signatures / units needed" → always round UP (ceiling).

**Reasoning style.**
- Solve directly using one clear method. Commit to your approach.
- Keep steps focused and concise; most problems take 200-600 tokens of reasoning.
- Long, exploratory reasoning is a sign of going off-track, not of being thorough.
- Read the problem carefully once at the start — especially units, limits of integration, and what is being asked for.

**Verification.**
- After reaching your answer, do ONE quick sanity check (units, sign, or magnitude).
- Then commit. Do not re-derive, restart, or explore alternative methods.
- If a check reveals a clear error, fix that specific step and continue forward.

**No matching option (multiple choice).**
- If your computed answer doesn't match any option exactly, output \\boxed{NONE}.
- Do NOT pick the closest option.
- Do NOT reinterpret the problem, invent typos, or assume the problem meant something different to force a match.
- Trust your computation over the answer choices.

**Worked examples of correct final answers:**
- Sum of first 325 positive even numbers → \\boxed{325*(1+325)}
- 145°F to Celsius → \\boxed{62.7777777777778}
- All solutions to tan(θ)=4.76 in form θ=a+b·n → \\boxed{atan(4.76), pi}
- Fraction remaining after 36 years, half-life 31 → \\boxed{(1/2)^(36/31)}
- Half-life when daily decay is 3.416% → \\boxed{[ln(0.5)]/[ln(0.96584)]}
- Standard deviation when variance is 60.8 → \\boxed{7.79743547584717}
- Reduce 25/40 → \\boxed{5/8}
- Integral with no matching multiple-choice option → \\boxed{NONE}"""

SYSTEM_PROMPT_MCQ = """You are an expert mathematician. Read the problem and the options, then output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}.

**Procedure:**
1. Solve the problem independently first, without looking at the options.
2. Then match your derived answer to the options.
3. **If your computed answer doesn't match any option, do NOT pick the closest one.** Re-check your computation — you likely made an arithmetic or sign error, or misread the problem. Also check whether an option is mathematically equivalent to your answer in a different form (e.g., your `pi*sqrt(a)` might equal an option written as `sqrt(a)*pi`).
4. Never collapse a symbolic answer (with π, e, √, etc.) to the closest decimal option just because the options are all decimals — that usually means you misread the problem or the options. Re-examine first.

**Reasoning style.** Solve directly, no "this is complex" preambles. Verify briefly. If you spot an error, fix it and move on without dwelling.
"""

In [ ]:
# Cell 5 — Hugging Face login (optional; set HF_TOKEN in Colab secrets)
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(hf_token)
        print("HF login successful.")
    else:
        print("No HF_TOKEN secret found; continuing.")
except Exception as e:
    print("HF login skipped:", e)

In [ ]:
# Cell 6 — Split public.jsonl (926 train / 200 val, stratified MCQ + free)

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def is_mcq_row(row):
    return bool(row.get("options"))


def stratified_val_split(rows, val_size=200, seed=42):
    rng = random.Random(seed)
    mcq = [r for r in rows if is_mcq_row(r)]
    free = [r for r in rows if not is_mcq_row(r)]
    n = len(rows)

    if val_size > n:
        raise ValueError(f"val_size={val_size} > total={n}")

    val_mcq_target = round(val_size * len(mcq) / n)
    val_mcq_target = max(0, min(val_mcq_target, len(mcq), val_size))
    val_free_target = val_size - val_mcq_target

    if val_free_target > len(free):
        val_free_target = len(free)
        val_mcq_target = val_size - val_free_target
    if val_mcq_target > len(mcq):
        val_mcq_target = len(mcq)
        val_free_target = val_size - val_mcq_target

    rng.shuffle(mcq)
    rng.shuffle(free)

    val_rows = mcq[:val_mcq_target] + free[:val_free_target]
    val_ids = {r["id"] for r in val_rows}
    train_rows = [r for r in rows if r["id"] not in val_ids]

    rng.shuffle(val_rows)
    rng.shuffle(train_rows)

    stats = {
        "total": n,
        "mcq_total": len(mcq),
        "free_total": len(free),
        "val_mcq": val_mcq_target,
        "val_free": val_free_target,
        "train_mcq": sum(is_mcq_row(r) for r in train_rows),
        "train_free": sum(not is_mcq_row(r) for r in train_rows),
    }
    return train_rows, val_rows, stats


def gold_to_boxed(answer, mcq: bool) -> str:
    if mcq:
        return f"\\boxed{{{str(answer).strip().upper()}}}"
    if isinstance(answer, list):
        inner = ", ".join(str(x) for x in answer)
        return f"\\boxed{{{inner}}}"
    return f"\\boxed{{{answer}}}"


def row_to_sft_messages(row):
    opts = row.get("options")
    mcq = bool(opts)
    user = row["question"]
    if mcq:
        labels = [chr(65 + i) for i in range(len(opts))]
        opts_text = "\n".join(f"{lbl}. {o}" for lbl, o in zip(labels, opts))
        user = f"{user}\n\nOptions:\n{opts_text}"

    assistant = gold_to_boxed(row["answer"], mcq)
    return {
        "id": row["id"],
        "is_mcq": mcq,
        "messages": [
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ],
    }


all_rows = load_jsonl(PUBLIC_PATH)
train_rows, val_rows, split_stats = stratified_val_split(all_rows, val_size=VAL_SIZE, seed=SEED)

write_jsonl(TRAIN_RAW_PATH, train_rows)
write_jsonl(VAL_RAW_PATH, val_rows)

train_sft_rows = [row_to_sft_messages(r) for r in train_rows]
val_sft_rows = [row_to_sft_messages(r) for r in val_rows]
write_jsonl(TRAIN_SFT_PATH, train_sft_rows)
write_jsonl(VAL_SFT_PATH, val_sft_rows)

split_meta = {
    "seed": SEED,
    "val_size": VAL_SIZE,
    "stats": split_stats,
    "train_ids": sorted(r["id"] for r in train_rows),
    "val_ids": sorted(r["id"] for r in val_rows),
}
with open(SPLIT_META_PATH, "w", encoding="utf-8") as f:
    json.dump(split_meta, f, indent=2)

print("Split stats:", split_stats)
print("Train raw:", len(train_rows), TRAIN_RAW_PATH)
print("Val raw:  ", len(val_rows), VAL_RAW_PATH)
print("Train SFT:", len(train_sft_rows), TRAIN_SFT_PATH)
print("Val SFT:  ", len(val_sft_rows), VAL_SFT_PATH)
print("Example assistant target:", train_sft_rows[0]["messages"][1]["content"])

In [ ]:
# Cell 7 — Load tokenizer + base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print("Model + tokenizer loaded.")

In [ ]:
# Cell 8 — Build HF datasets (prompt/completion format)
# Works with Qwen3-Thinking chat template; avoids assistant_only_loss template error.

def row_to_prompt_completion(row):
    user_msg = row["messages"][0]["content"]
    assistant_msg = row["messages"][1]["content"]
    mcq = row.get("is_mcq", False)
    system_prompt = SYSTEM_PROMPT_MCQ if mcq else SYSTEM_PROMPT_MATH

    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ],
        "completion": [
            {"role": "assistant", "content": assistant_msg},
        ],
    }


train_ds = Dataset.from_list([row_to_prompt_completion(r) for r in train_sft_rows])
val_ds = Dataset.from_list([row_to_prompt_completion(r) for r in val_sft_rows])

print("Train dataset:", len(train_ds))
print("Val dataset:", len(val_ds))
print("Sample keys:", train_ds.column_names)
print("Prompt:", train_ds[0]["prompt"][-1]["content"][:200])
print("Completion:", train_ds[0]["completion"][0]["content"])

In [ ]:
# Cell 9 — LoRA config (conservative)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
print(peft_config)

In [ ]:
# Cell 10 — Training config
# Use completion_only_loss (NOT assistant_only_loss) for Qwen3-Thinking templates.
training_args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    bf16=True,
    fp16=False,
    max_length=2048,
    packing=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
    completion_only_loss=True,
)

print(training_args)

In [ ]:
# Cell 11 — Train
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

train_result = trainer.train()
print(train_result)

In [ ]:
# Cell 12 — Save adapter
trainer.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Saved LoRA adapter to:", OUT_DIR)

In [ ]:
# Cell 13 — Quick generation sanity check (extract boxed answer)

def extract_last_boxed(text: str):
    matches = re.findall(r"\\boxed\{(.*?)\}", text, flags=re.DOTALL)
    return matches[-1].strip() if matches else None


USER_SUFFIX = "\n\nSolve briefly. End with exactly one \\boxed{...}. No text after the box."

for i, row in enumerate(val_sft_rows[:3], start=1):
    user_msg = row["messages"][0]["content"]
    mcq = row["is_mcq"]
    system_prompt = SYSTEM_PROMPT_MCQ if mcq else SYSTEM_PROMPT_MATH

    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg + USER_SUFFIX},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        out = trainer.model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    boxed = extract_last_boxed(completion)
    gold = row["messages"][1]["content"]

    print(f"\n=== VAL SAMPLE {i} (id={row['id']}) ===")
    print("GOLD:", gold)
    print("PRED BOXED:", boxed)
    print("RAW (first 400 chars):", completion[:400])

In [ ]:
# Cell 14 — Optional: score all 200 val items with judger (copy judger.py to Drive project)
import sys
sys.path.insert(0, PROJECT_ROOT)

try:
    from judger import Judger
    from tqdm import tqdm

    judger = Judger(strict_extract=False)

    def score_free(pred, gold_list):
        try:
            return judger.auto_judge(
                pred=pred,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            return False

    val_gold_by_id = {r["id"]: r for r in val_rows}
    print("Judger loaded. Run full val inference in vLLM notebook, then score responses here.")
except Exception as e:
    print("Judger not available in this environment:", e)